In [1]:
# import necessary libraries
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

import joblib

from pathlib import Path

In [2]:
hotels = pd.read_csv("../data/raw/hotels.csv")

hotels.head()

,travelCode,userCode,name,place,days,price,total,date
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,11/14/2019
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,12/12/2019
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,12/26/2019


In [4]:
hotels.columns

Index(['travelCode', 'userCode', 'name', 'place', 'days', 'price', 'total',
       'date'],
      dtype='object')

In [5]:
hotels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   travelCode  40552 non-null  int64  
 1   userCode    40552 non-null  int64  
 2   name        40552 non-null  object 
 3   place       40552 non-null  object 
 4   days        40552 non-null  int64  
 5   price       40552 non-null  float64
 6   total       40552 non-null  float64
 7   date        40552 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 2.5+ MB


In [6]:
hotels.isnull().sum()

travelCode    0
userCode      0
name          0
place         0
days          0
price         0
total         0
date          0
dtype: int64

In [21]:
# create interaction matrix
interaction_matrix = pd.pivot_table(
    hotels,
    index="userCode",
    columns="name",
    values="travelCode",
    aggfunc="count",
    fill_value=0
)

interaction_matrix.head()

name,Hotel A,Hotel AF,Hotel AU,Hotel BD,Hotel BP,Hotel BW,Hotel CB,Hotel K,Hotel Z
userCode,,,,,,,,,
0,3,4,2,4,1,2,1,7,3
1,0,1,0,0,1,0,0,0,0
2,6,2,3,2,2,7,3,5,6
3,10,6,7,11,2,9,7,6,2
4,7,6,7,5,3,6,11,7,4


In [22]:
# compute user similarity matrix
similarity = cosine_similarity(interaction_matrix)

user_similarity = pd.DataFrame(
    similarity,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

user_similarity.head()

userCode,0,1,2,3,4,5,6,7,8,9,...,1330,1331,1332,1333,1334,1335,1336,1337,1338,1339
userCode,,,,,,,,,,,,,,,,,,,,,
0,1.000000,0.338643,0.808627,0.817538,0.805124,0.920250,0.808229,0.517285,0.775270,0.789883,...,0.705327,0.691148,0.813617,0.799943,0.312825,0.727077,0.095783,0.794256,0.095783,0.818854
1,0.338643,1.000000,0.213201,0.258199,0.322252,0.339683,0.636446,0.218218,0.297318,0.346410,...,0.473365,0.744092,0.477334,0.444500,0.866025,0.449977,0.707107,0.308607,0.707107,0.442326
2,0.808627,0.213201,1.000000,0.842925,0.858804,0.663856,0.678454,0.697863,0.788392,0.910877,...,0.711039,0.645895,0.809904,0.764912,0.276956,0.781188,0.150756,0.657952,0.150756,0.817303
3,0.817538,0.258199,0.842925,1.000000,0.917567,0.635867,0.842189,0.767682,0.935599,0.816165,...,0.663889,0.638125,0.760024,0.782892,0.316776,0.757268,0.091287,0.737058,0.091287,0.761387
4,0.805124,0.322252,0.858804,0.917567,1.000000,0.697323,0.848869,0.648517,0.838347,0.847571,...,0.711865,0.723160,0.791897,0.823067,0.392777,0.902259,0.151911,0.707193,0.151911,0.781332


In [ ]:
# save the interaction matrix and user similarity matrix to disk
joblib.dump(
    interaction_matrix,
    "../models/user_hotel_matrix.pkl"
)

joblib.dump(
    user_similarity,
    "../models/user_similarity.pkl"
)

print("Recommendation model saved successfully.")

Recommendation model saved successfully.


In [24]:
# recommendation function
def recommend_hotels(user_id, top_n=5):

    if user_id not in interaction_matrix.index:
        return []

    similar_users = (
        user_similarity[user_id]
        .sort_values(ascending=False)
        .drop(user_id)
    )

    recommendations = pd.Series(dtype=float)

    for similar_user in similar_users.index[:10]:

        recommendations = recommendations.add(
            interaction_matrix.loc[similar_user],
            fill_value=0
        )

    visited = interaction_matrix.loc[user_id]

    recommendations = recommendations[visited == 0]

    # If user has already visited every hotel,
    # show their own highest-booked hotels instead.
    if recommendations.empty:

        return (
            visited.sort_values(ascending=False)
            .head(top_n)
            .index.tolist()
        )

    return (
        recommendations
        .sort_values(ascending=False)
        .head(top_n)
        .index.tolist()
    )

In [25]:
recommend_hotels(0)

['Hotel K', 'Hotel BD', 'Hotel AF', 'Hotel A', 'Hotel Z']